# SynthFace on Google Colab: 3D synthetic identities (approach A)

Renders many synthetic identities with **Blender 5.2.2 + MPFB 2.0.17** (same versions and assets as the dev
laptop, every download sha256-checked) on the Colab **GPU**. Same seed gives the same identities as on the laptop.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU` (or better).
2. Nothing to set up for the code: it is cloned from the public GitHub repo (no token).
3. Run the cells top to bottom. Output goes to Google Drive (`DRIVE_DIR`) so a disconnect loses nothing;
   re-running the render cell resumes where it stopped.

Rules (CLAUDE.md): license-clean assets only (all already in `assets/LICENSES.csv`), no AI-generated faces here,
never commit `output/`.

## 1. Settings

In [ ]:
JOB_ID = "colab_mix20_v4"       # NEW name for every new batch / code version (an existing name resumes)
SEED = 20261013                 # same seed + same settings = same identities
# Which people: a mix of groups (tools/pick_identities.py picks matching identities of SEED),
# or MIX = None for identities 1..NUM_IDENTITIES. Groups: female, male_clean, male_beard,
# male_stubble, male_short, male_full, male_goatee.
MIX = "female=8,male_clean=4,male_stubble=2,male_short=2,male_full=2,male_goatee=2"
NUM_IDENTITIES = 20             # only used when MIX = None
REJECT_NEAR_DUPLICATES = True   # skip identities whose face is practically the same as one already chosen
                                # (blender/select_identities.py, ~2-3 s per candidate on the CPU)
                                # Also compared with every earlier selection in output/selections/ (on Drive),
                                # so separate batches never share a face.
IMAGES_PER_IDENTITY = 10
SAMPLES = 48                    # Cycles samples (+ denoiser)
DEVICE = "OPTIX"                # "OPTIX" (T4/L4/A100), "CUDA" if OptiX fails, "CPU" without GPU
REPO = "https://github.com/ankitscode/-synthetic-Image-Genrator.git"   # public, no token needed
BRANCH = "master"               # master = 3D pipeline
DRIVE_DIR = "/content/drive/MyDrive/synthface_output"   # None = keep output only in this VM

PROJECT = "/content/synthface"
TOOLS = "/content/synthface_tools"
BLENDER = f"{TOOLS}/blender-5.2.2-linux-x64/blender"

## 2. Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU (or set DEVICE = 'CPU', very slow)"

## 3. Mount Drive and clone the public GitHub repo

In [ ]:
import os, subprocess

if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
print("Code version:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %cd %s", "--date=short"],
                                      capture_output=True, text=True).stdout)


## 4. Install Blender, MPFB, asset packs and HDRIs (about 3-5 min)

In [ ]:
!bash {PROJECT}/colab/setup_colab.sh {TOOLS}

## 5. Send output to Drive (optional)

In [ ]:
out = f"{PROJECT}/output"
if DRIVE_DIR and not os.path.islink(out):
    if os.path.isdir(out) and os.listdir(out):
        raise SystemExit(f"{out} already has files; move them first")
    if os.path.isdir(out):
        os.rmdir(out)
    os.symlink(DRIVE_DIR, out)
os.makedirs(out, exist_ok=True)
print(out, "->", os.path.realpath(out))

## 6. Render the identities

Prints one line per finished identity. Safe to re-run after a disconnect (`--resume` skips finished identities).
Blender's log is in `output/<JOB_ID>/blender_stderr.log`.

In [ ]:
import json, time

job_dir = f"{PROJECT}/output/{JOB_ID}"
os.makedirs(job_dir, exist_ok=True)
import sys
if REJECT_NEAR_DUPLICATES:
    selection = f"{PROJECT}/output/selections/{JOB_ID}.json"
    if not os.path.exists(selection):
        import glob
        earlier = sorted(glob.glob(f"{PROJECT}/output/selections/*.json"))
        sel = [BLENDER, "--background", "--python", "blender/select_identities.py", "--", "--seed", str(SEED),
               "--out", selection] + (["--mix", MIX] if MIX else ["--count", str(NUM_IDENTITIES)])
        if earlier:
            sel += ["--against", *earlier]
            print(f"Checking against {len(earlier)} earlier selection(s)")
        r = subprocess.run(sel, cwd=PROJECT, capture_output=True, text=True)
        print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("AGAINST", "REJECT", "ACCEPTED", "SELECTION"))))
        if r.returncode != 0 or not os.path.exists(selection):
            raise RuntimeError("identity selection failed:\n" + r.stderr[-2000:])
    indices = json.load(open(selection))["accepted"]
elif MIX:
    sys.path.insert(0, f"{PROJECT}/tools")
    from pick_identities import parse_mix, pick
    chosen = pick(SEED, parse_mix(MIX))
    for c in chosen:
        print(f"id_{c['index']:05d}  {c['group']:<12} {c['age_years']:>4.0f}y  hair {c['hair']}")
    indices = [c["index"] for c in chosen]
else:
    indices = list(range(1, NUM_IDENTITIES + 1))
print(f"{len(indices)} identities: {indices[:30]}{' ...' if len(indices) > 30 else ''}")
job = {"job_id": JOB_ID, "seed": SEED, "num_identities": len(indices), "identity_indices": indices,
       "images_per_identity": IMAGES_PER_IDENTITY, "render": {"samples": SAMPLES, "device": DEVICE}}
job_path = f"{job_dir}/job.json"
if os.path.exists(job_path):
    if json.load(open(job_path)) != job:
        raise SystemExit(f"{job_path} exists with other settings: use a new JOB_ID")
else:
    json.dump(job, open(job_path, "w"), indent=2)

cmd = [BLENDER, "--background", "--python", "blender/render_batch.py", "--", "--job", job_path, "--resume"]
t0 = time.time()
with open(f"{job_dir}/blender_stderr.log", "a") as err:
    proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=err, text=True)
    for line in proc.stdout:
        if line.startswith('{"event"'):
            ev = json.loads(line)
            note = " (already done)" if ev.get("skipped") else ""
            done = indices.index(ev["index"]) + 1
            print(f"id_{ev['index']:05d} ({done}/{len(indices)}) {ev['status']}{note}  {time.time() - t0:.0f}s", flush=True)
print("Blender exit code:", proc.wait())

## 7. Contact sheet (one row per identity)

In [ ]:
!cd {PROJECT} && python tools/contact_sheet.py output/{JOB_ID} --max-cols 10
from IPython.display import Image, display
display(Image(f"{job_dir}/contact_sheet.png"))

## 8. Bring the batch back to the laptop

- With Drive: it is already in `MyDrive/synthface_output/<JOB_ID>/`. Download that folder.
- Without Drive: run the cell below to download a zip.

On the laptop, unzip to `output/<JOB_ID>/` in the project, then review with
`venv/bin/python tools/contact_sheet.py output/<JOB_ID>`.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{JOB_ID}", "zip", job_dir)
files.download(zip_path)